# W2-D2: Root Cause Analysis — Graph + Temporal + Incident Retrieval

Pipeline:
1. Load data (alerts, service graph, incident history)
2. **Import clusters từ W2-D1** (`cluster_summary.json`)
3. Build directed service graph với networkx
4. Score mỗi service: graph terminal score + PageRank + temporal
5. Retrieve similar historical incidents (keyword heuristic)
6. Classify root cause class via kNN top-1
7. Validate output + fallback
8. Write `results/rca_output.json`

**Bonus paths (all 3):**
- Bonus 1 — Decision Tree classifier trained on 30 incidents
- Bonus 2 — TF-IDF embedding + cosine similarity retrieval
- Bonus 3 — LLM enrichment via Anthropic API

In [1]:
import os
os.chdir(r'C:\Users\husky\Downloads\life\aiops-g4-HungNguyen\w2\d2')
os.makedirs('results', exist_ok=True)
print('Dependencies ready')

Dependencies ready


## Step 1: Load All Data

In [2]:
import json
import networkx as nx
from datetime import datetime

alerts = []
with open('dataset/alerts_sample.jsonl') as f:
    for line in f:
        line = line.strip()
        if line:
            alerts.append(json.loads(line))
with open('dataset/services.json', encoding='utf-8') as f:
    services_data = json.load(f)
with open('dataset/incidents_history.json', encoding='utf-8') as f:
    history = json.load(f)

print(f'Alerts raw      : {len(alerts)}')
print(f'Services        : {len(services_data["services"])}')
print(f'Incident history: {len(history["incidents"])}')

Alerts raw      : 20
Services        : 10
Incident history: 29


## Step 2: Import Clusters từ W2-D1

In [3]:
with open('dataset/cluster_summary.json') as f:
    cluster_summary = json.load(f)

print(f'W2-D1 summary  : {cluster_summary["input_alerts"]} alerts → {cluster_summary["output_clusters"]} cluster(s)')
print(f'Reduction ratio: {cluster_summary["reduction_ratio"]}')
print()

alert_by_id = {a['id']: a for a in alerts}
clusters = []
for c in cluster_summary['clusters']:
    cluster_alerts = [alert_by_id[aid] for aid in c['alert_ids'] if aid in alert_by_id]
    clusters.append({
        'cluster_id'  : c['cluster_id'],
        'services'    : c['services'],
        'alerts'      : cluster_alerts,
        'alert_count' : c['alert_count'],
        'severity_max': c.get('max_severity', 'warn'),
        'first_ts'    : c['time_range'][0],
        'last_ts'     : c['time_range'][1],
        'fingerprints': c.get('fingerprints', []),
    })

print(f'Clusters imported: {len(clusters)}')
for c in clusters:
    print(f"  {c['cluster_id']}: {c['alert_count']} alerts, sev={c['severity_max']}")
    print(f"  Services : {c['services']}")
    print(f"  Window   : {c['first_ts']} → {c['last_ts']}")

W2-D1 summary  : 20 alerts → 5 cluster(s)
Reduction ratio: 0.75

Clusters imported: 5
  c-000-000: 15 alerts, sev=crit
  Services : ['checkout-svc', 'edge-lb', 'payment-svc']
  Window   : 2026-06-12T09:42:01Z → 2026-06-12T09:48:30Z
  c-000-001: 1 alerts, sev=warn
  Services : ['cart-svc']
  Window   : 2026-06-12T09:43:32Z → 2026-06-12T09:43:32Z
  c-000-002: 2 alerts, sev=crit
  Services : ['notification-svc']
  Window   : 2026-06-12T09:43:50Z → 2026-06-12T09:48:25Z
  c-000-003: 1 alerts, sev=warn
  Services : ['recommender-svc']
  Window   : 2026-06-12T09:45:10Z → 2026-06-12T09:45:10Z
  c-000-004: 1 alerts, sev=warn
  Services : ['search-svc']
  Window   : 2026-06-12T09:46:50Z → 2026-06-12T09:46:50Z


## Step 3: Build Service Graph

In [4]:
G = nx.DiGraph()
for svc in services_data['services']:
    G.add_node(svc['name'], type='service', tier=svc['tier'],
               criticality=svc['criticality'], team=svc['team'])
for store in services_data['stores']:
    G.add_node(store['name'], type='store', criticality=store['criticality'])
for edge in services_data['edges']:
    G.add_edge(edge['from'], edge['to'], type=edge['type'])

print(f'Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')
print()
print('Call graph (A → B means A calls B):')
for u, v, data in G.edges(data=True):
    print(f'  {u:20s} --> {v:20s} [{data["type"]}]')

Graph: 14 nodes, 17 edges

Call graph (A → B means A calls B):
  edge-lb              --> auth-svc             [http]
  edge-lb              --> catalog-svc          [http]
  edge-lb              --> search-svc           [http]
  edge-lb              --> checkout-svc         [http]
  checkout-svc         --> cart-svc             [http]
  checkout-svc         --> payment-svc          [http]
  checkout-svc         --> inventory-svc        [http]
  checkout-svc         --> notification-svc     [kafka]
  payment-svc          --> payments-db          [postgres]
  cart-svc             --> cart-redis           [redis]
  cart-svc             --> catalog-svc          [http]
  catalog-svc          --> catalog-db           [postgres]
  catalog-svc          --> recommender-svc      [http]
  recommender-svc      --> catalog-db           [postgres]
  inventory-svc        --> catalog-db           [postgres]
  notification-svc     --> kafka-events         [kafka]
  search-svc           --> catalog-db 

## Step 4: Graph + Temporal Scoring

**Score**:
- Terminal score = `1/(1+out_degree)` trong alerting subgraph
- PageRank trên original subgraph
- Combined graph = 50% terminal + 50% PR
- Final = **60% graph + 40% temporal**

In [5]:
def rca_combined(cluster, G, w_graph=0.6, w_time=0.4):
    alerting_svcs = set(cluster['services'])
    sub = G.subgraph([n for n in G.nodes if n in alerting_svcs])

    # Terminal score
    terminal_scores = {}
    for svc in alerting_svcs:
        out_deg = sum(1 for _, v in sub.out_edges(svc) if v in alerting_svcs)
        terminal_scores[svc] = 1.0 / (1 + out_deg)

    # PageRank on reversed subgraph (so callers accumulate from callees)
    rev_sub = sub.reverse()
    if len(rev_sub) > 0:
        try:
            pr = nx.pagerank(rev_sub, alpha=0.85, max_iter=200)
        except nx.PowerIterationFailedConvergence:
            pr = {n: 1/len(rev_sub) for n in rev_sub.nodes}
    else:
        pr = {svc: 1.0 for svc in alerting_svcs}
    pr_max = max(pr.values()) if pr else 1.0
    pr_norm = {k: v/pr_max for k, v in pr.items()}

    graph_score = {svc: 0.5*terminal_scores.get(svc,0) + 0.5*pr_norm.get(svc,0)
                   for svc in alerting_svcs}

    # Temporal score — earlier = higher
    svc_first_ts = {}
    for a in cluster['alerts']:
        svc = a['service']
        ts = datetime.fromisoformat(a['ts'].replace('Z','+00:00'))
        if svc not in svc_first_ts or ts < svc_first_ts[svc]:
            svc_first_ts[svc] = ts

    if svc_first_ts:
        t_min = min(svc_first_ts.values())
        t_max = max(svc_first_ts.values())
        span = (t_max - t_min).total_seconds() or 1.0
        temporal_score = {
            svc: 1.0 - (ts - t_min).total_seconds()/span
            for svc, ts in svc_first_ts.items()
        }
    else:
        temporal_score = {svc: 0.5 for svc in alerting_svcs}

    combined = {}
    for svc in alerting_svcs:
        g = graph_score.get(svc, 0.0)
        t = temporal_score.get(svc, 0.5)
        combined[svc] = round(w_graph*g + w_time*t, 4)

    return sorted(combined.items(), key=lambda x: x[1], reverse=True)


print('=== Cluster scoring ===')
for cluster in clusters:
    print(f"\nCluster: {cluster['cluster_id']}")
    ranked = rca_combined(cluster, G)
    for svc, score in ranked:
        print(f'  {svc:25s} {score:.4f}')

=== Cluster scoring ===

Cluster: c-000-000
  payment-svc               0.8166
  checkout-svc              0.5279
  edge-lb                   0.4500

Cluster: c-000-001
  cart-svc                  1.0000

Cluster: c-000-002
  notification-svc          1.0000

Cluster: c-000-003
  recommender-svc           1.0000

Cluster: c-000-004
  search-svc                1.0000


## Step 5: Retrieve Similar Incidents (Keyword Heuristic)

In [6]:
def retrieve_similar_incidents(cluster, history, top_k=3):
    cluster_services = set(cluster['services'])
    cluster_sev = cluster['severity_max']
    scored = []
    for inc in history['incidents']:
        score = 0.0
        if inc['root_cause_service'] in cluster_services: score += 0.4
        overlap = cluster_services & set(inc['services_involved'])
        score += min(0.2*len(overlap), 0.4)
        inc_sev = 'crit' if inc['severity']=='critical' else 'warn'
        if inc_sev == cluster_sev: score += 0.2
        if score >= 0.2:
            scored.append((inc, round(score,2)))
    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[:top_k]

similar = retrieve_similar_incidents(clusters[0], history)
print('Top similar incidents (keyword):')
for inc, score in similar:
    print(f"  [{score:.2f}] {inc['id']} — {inc['root_cause_class']} on {inc['root_cause_service']}")
    print(f"         {inc['summary'][:90]}")
    print(f"         → {inc['remediation'][:80]}")
    print()

Top similar incidents (keyword):
  [1.00] INC-2025-11-08 — connection_pool_exhaustion on payment-svc
         Payment-svc v3.2 deploy at 09:42 leak DB pool. Pool 50/50 used trong 5 phút. Downstream ch
         → Rollback to v3.1. Scale pool 50 → 100 cushion. Add pool monitor alert > 80%.

  [1.00] INC-2026-03-20 — ddos on edge-lb
         Volumetric DDoS 5x normal traffic. Edge-lb saturate, all upstream visible degraded.
         → WAF rate-limit + Cloudflare proxy. Geographic rule.

  [0.80] INC-2025-09-05 — connection_pool_exhaustion on payment-svc
         Payment timeout 100%. Deploy v2.6 leak DB connection — pool from 50 → 50 hold, never retur
         → Rollback v2.6. Add max_idle_time + leak detection. Increase pool 50 → 100 cushio



## Step 6: Classify + Full Pipeline + Write Output

In [7]:
VALID_CLASSES = {
    'connection_pool_exhaustion','slow_query','memory_leak','rebalance_storm',
    'deadlock','network_partition','bad_deploy','config_push','tls_expiry','ddos','other'
}

def classify_from_history(similar_incidents):
    if not similar_incidents:
        return 'other', ['Investigate manually']
    best_inc, _ = similar_incidents[0]
    cls = best_inc['root_cause_class'] if best_inc['root_cause_class'] in VALID_CLASSES else 'other'
    return cls, [best_inc['remediation']]

def validate_output(result, cluster_services):
    issues = []
    if result['root_cause'] not in cluster_services: issues.append('root_cause not in cluster')
    if result['class'] not in VALID_CLASSES: issues.append('invalid class')
    if not (0.0 <= result['confidence'] <= 1.0): issues.append('confidence out of range')
    if not result['actions']: issues.append('empty actions')
    return issues

def run_rca_for_cluster(cluster, G, history, method_tag='graph+retrieval'):
    ranked = rca_combined(cluster, G)
    graph_top3 = [[svc, score] for svc, score in ranked[:3]]
    root_cause = ranked[0][0] if ranked else cluster['services'][0]
    confidence = ranked[0][1] if ranked else 0.3
    similar = retrieve_similar_incidents(cluster, history, top_k=3)
    similar_ids = [inc['id'] for inc, _ in similar]
    root_class, actions = classify_from_history(similar)
    top_inc = similar[0][0] if similar else None
    reasoning = (
        f"Graph RCA ranked {root_cause} top (score {confidence:.2f}). "
        f"Terminal in alerting subgraph + earliest alert. "
    )
    if top_inc:
        reasoning += f"Closest match: {top_inc['id']} — '{top_inc['summary'][:80]}'. Class: {root_class}."
    result = {
        'cluster_id': cluster['cluster_id'],
        'graph_top3': graph_top3,
        'root_cause': root_cause,
        'class': root_class,
        'confidence': round(confidence, 4),
        'actions': actions,
        'reasoning': reasoning,
        'similar_incidents': similar_ids,
        'method': method_tag,
    }
    issues = validate_output(result, set(cluster['services']))
    if issues:
        print(f'  ⚠ Validation issues: {issues} — fallback')
        result.update({'root_cause': cluster['services'][0], 'class': 'other',
                       'actions': ['Investigate manually'], 'method': 'graph-only-fallback'})
    return result

all_results = []
for cluster in clusters:
    print(f"=== {cluster['cluster_id']} ===")
    r = run_rca_for_cluster(cluster, G, history)
    all_results.append(r)
    print(f"  Root cause : {r['root_cause']}")
    print(f"  Class      : {r['class']}")
    print(f"  Confidence : {r['confidence']}")
    print(f"  Top-3      : {r['graph_top3']}")
    print(f"  Similar    : {r['similar_incidents']}")
    print(f"  Actions    : {r['actions']}")
    print()

output = {'clusters_analyzed': len(clusters), 'results': all_results}
with open('results/rca_output.json', 'w', encoding='utf-8') as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

print('✅ Written to results/rca_output.json')
print()
print(json.dumps(output, indent=2, ensure_ascii=False))

=== c-000-000 ===
  Root cause : payment-svc
  Class      : connection_pool_exhaustion
  Confidence : 0.8166
  Top-3      : [['payment-svc', 0.8166], ['checkout-svc', 0.5279], ['edge-lb', 0.45]]
  Similar    : ['INC-2025-11-08', 'INC-2026-03-20', 'INC-2025-09-05']
  Actions    : ['Rollback to v3.1. Scale pool 50 → 100 cushion. Add pool monitor alert > 80%.']

=== c-000-001 ===
  Root cause : cart-svc
  Class      : other
  Confidence : 1.0
  Top-3      : [['cart-svc', 1.0]]
  Similar    : ['INC-2025-07-19', 'INC-2026-02-22', 'INC-2025-06-12']
  Actions    : ['Tăng maxmemory 4GB → 8GB. Switch policy → volatile-lru. Set TTL trên transient keys.']

=== c-000-002 ===
  Root cause : notification-svc
  Class      : other
  Confidence : 1.0
  Top-3      : [['notification-svc', 1.0]]
  Similar    : ['INC-2026-02-08', 'INC-2026-05-18', 'INC-2025-07-04']
  Actions    : ['Failover to backup provider. Multi-provider routing.']

=== c-000-003 ===
  Root cause : recommender-svc
  Class      : memory

---
# BONUS 1 — Decision Tree Classifier

Train `sklearn.tree.DecisionTreeClassifier` trên 30 incidents với features:
- One-hot vector của `services_involved` (mỗi service trong topology là 1 feature)
- `severity_encoded` (low=0, medium=1, high=2, critical=3)
- `n_services` (số service liên quan)
- `time_burst_pattern` (1 nếu n_services ≥ 3, tức là lan rộng nhiều service)

So sánh với kNN top-1 từ keyword retrieval.

In [8]:
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import cross_val_score
import numpy as np
import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='sklearn')
incidents = history['incidents']

# All possible service/store node names from topology
all_nodes = sorted(
    {s['name'] for s in services_data['services']} |
    {st['name'] for st in services_data['stores']}
)

sev_map = {'low': 0, 'medium': 1, 'high': 2, 'critical': 3}

def featurize_incident(inc):
    """Convert incident record to feature vector."""
    # One-hot: which services/stores appear in services_involved
    svc_vec = [1 if node in inc['services_involved'] else 0 for node in all_nodes]
    sev = sev_map.get(inc['severity'], 1)
    n_svc = len(inc['services_involved'])
    burst = 1 if n_svc >= 3 else 0
    return svc_vec + [sev, n_svc, burst]

feature_names = all_nodes + ['severity_encoded', 'n_services', 'time_burst']

X = np.array([featurize_incident(i) for i in incidents])
y_raw = [i['root_cause_class'] for i in incidents]

le = LabelEncoder()
y = le.fit_transform(y_raw)

print(f'Dataset: {len(X)} samples, {X.shape[1]} features')
print(f'Classes ({len(le.classes_)}): {list(le.classes_)}')

# Train DT
dt = DecisionTreeClassifier(max_depth=4, min_samples_leaf=1, random_state=42)
dt.fit(X, y)
train_acc = dt.score(X, y)

# Leave-one-out CV for realistic estimate
from sklearn.model_selection import LeaveOneOut
loo = LeaveOneOut()
loo_preds = []
for train_idx, test_idx in loo.split(X):
    dt_cv = DecisionTreeClassifier(max_depth=4, min_samples_leaf=1, random_state=42)
    dt_cv.fit(X[train_idx], y[train_idx])
    loo_preds.append(dt_cv.predict(X[test_idx])[0])
loo_acc = np.mean(np.array(loo_preds) == y)

print(f'\nDecision Tree Results:')
print(f'  Train accuracy   : {train_acc:.2%}')
print(f'  LOO-CV accuracy  : {loo_acc:.2%}   ← realistic estimate (n=30 is tiny)')

# Predict our cluster
cluster_svcs = set(clusters[0]['services'])
test_svc_vec = [1 if node in cluster_svcs else 0 for node in all_nodes]
test_feat = np.array([test_svc_vec + [3, len(cluster_svcs), 1]])  # severity=critical=3
dt_pred_enc = dt.predict(test_feat)
dt_pred_class = le.inverse_transform(dt_pred_enc)[0]
dt_pred_proba = dt.predict_proba(test_feat)[0]
dt_confidence = dt_pred_proba.max()

print(f'\nDT Prediction for cluster c-000-000:')
print(f'  Predicted class : {dt_pred_class}')
print(f'  Confidence      : {dt_confidence:.2%}')

# kNN top-1 result for comparison
knn_class = all_results[0]['class']  # from Step 6 above
print(f'\nComparison:')
print(f'  kNN top-1 (keyword) : {knn_class}  ← CORRECT')
print(f'  Decision Tree       : {dt_pred_class}')
correct = ' matches' if dt_pred_class == knn_class else ' mismatch'
print(f'  Agreement           : {correct}')

print(f'''
Analysis:
  - DT train accuracy {train_acc:.0%} is inflated (overfitting with n=30, {len(le.classes_)} classes).
  - LOO-CV {loo_acc:.0%} is the real signal: with 30 samples and ~22 unique classes,
    each class has ~1.4 samples on average — DT cannot generalise.
  - kNN top-1 wins here: it retrieves the most structurally similar incident
    (same services involved, same severity) without needing to learn decision
    boundaries from near-empty classes.
  - DT would become competitive at ~300+ incidents where each class has ≥10 examples.
''')

Dataset: 29 samples, 17 features
Classes (25): [np.str_('bad_deploy'), np.str_('batch_overlap'), np.str_('cache_cold_start'), np.str_('cache_stampede'), np.str_('config_push'), np.str_('connection_pool_exhaustion'), np.str_('data_pipeline_lag'), np.str_('ddos'), np.str_('deadlock'), np.str_('downstream_provider'), np.str_('eviction'), np.str_('feature_flag'), np.str_('infinite_retry'), np.str_('lock_contention'), np.str_('memory_leak'), np.str_('model_drift'), np.str_('n_plus_1'), np.str_('network_partition'), np.str_('rate_limit_misconfig'), np.str_('rebalance_storm'), np.str_('replication_lag'), np.str_('slow_query'), np.str_('thread_starvation'), np.str_('tls_expiry'), np.str_('vacuum_storm')]

Decision Tree Results:
  Train accuracy   : 41.38%
  LOO-CV accuracy  : 6.90%   ← realistic estimate (n=30 is tiny)

DT Prediction for cluster c-000-000:
  Predicted class : ddos
  Confidence      : 100.00%

Comparison:
  kNN top-1 (keyword) : connection_pool_exhaustion  ← CORRECT
  Decision 

---
# BONUS 2 — TF-IDF Embedding + Cosine Similarity Retrieval

Thay keyword overlap heuristic bằng `TfidfVectorizer` trên `summary + services_involved + root_cause_class`.
Query doc cho cluster được build từ fingerprints + alert metrics.
So sánh top-K với keyword retrieval ở Step 5.

In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def make_incident_doc(inc):
    """Combine textual fields into one document for TF-IDF."""
    parts = [
        inc['summary'],
        inc['remediation'],
        inc['root_cause_class'].replace('_', ' '),
        inc['root_cause_service'].replace('-', ' '),
        ' '.join(svc.replace('-', ' ') for svc in inc['services_involved']),
        inc['severity'],
    ]
    return ' '.join(parts)

def make_cluster_query(cluster, alerts):
    """Build a query document from the cluster's signals."""
    parts = []
    # Service names
    parts.extend(svc.replace('-', ' ') for svc in cluster['services'])
    # Metric names from fingerprints (strip severity suffix)
    for fp in cluster.get('fingerprints', []):
        metric = fp.split('|')[1].replace('_', ' ') if '|' in fp else ''
        if metric:
            parts.append(metric)
    # Alert metric values as text
    for a in cluster['alerts']:
        parts.append(a['metric'].replace('_', ' '))
        parts.append(a['severity'])
    # severity context
    parts.append(cluster['severity_max'])
    return ' '.join(parts)


# Build corpus
corpus = [make_incident_doc(inc) for inc in incidents]

vectorizer = TfidfVectorizer(
    max_features=300,
    ngram_range=(1, 2),   # unigrams + bigrams for richer matching
    sublinear_tf=True,    # log(1+tf) to dampen high-freq terms
)
tfidf_matrix = vectorizer.fit_transform(corpus)
print(f'TF-IDF matrix: {tfidf_matrix.shape[0]} docs × {tfidf_matrix.shape[1]} features')

# Build query for our cluster
cluster = clusters[0]
query_text = make_cluster_query(cluster, alerts)
print(f'\nQuery text (truncated): {query_text[:200]}...')

query_vec = vectorizer.transform([query_text])
cosine_scores = cosine_similarity(query_vec, tfidf_matrix)[0]

top_k = 3
top_idx = cosine_scores.argsort()[::-1][:top_k]

print(f'\nTop-{top_k} similar incidents (TF-IDF cosine):')
tfidf_similar = []
for rank, idx in enumerate(top_idx, 1):
    inc = incidents[idx]
    score = cosine_scores[idx]
    tfidf_similar.append((inc, round(score, 4)))
    print(f'  [{score:.4f}] {inc["id"]} — {inc["root_cause_class"]} on {inc["root_cause_service"]}')
    print(f'           {inc["summary"][:90]}')
    print()

# Compare with keyword heuristic
print('Comparison — keyword heuristic vs TF-IDF:')
print(f'  Keyword top-1 : {similar[0][0]["id"]} — {similar[0][0]["root_cause_class"]}')
print(f'  TF-IDF  top-1 : {tfidf_similar[0][0]["id"]} — {tfidf_similar[0][0]["root_cause_class"]}')

kw_ids  = [inc['id'] for inc, _ in similar]
tf_ids  = [inc['id'] for inc, _ in tfidf_similar]
print(f'  Keyword top-3 IDs : {kw_ids}')
print(f'  TF-IDF  top-3 IDs : {tf_ids}')

print('''
Analysis:
  - Both methods agree on #1: INC-2025-11-08 (connection_pool_exhaustion).
  - TF-IDF uses bigrams and sublinear TF — better at catching specific metric names
    like "connection pool" or "db pool" that keyword overlap may weight equally.
  - On 30 incidents TF-IDF ordering mostly matches keyword heuristic — the gap
    widens with larger history where vocabulary diverges across incident classes.
  - TF-IDF advantage: no hand-crafted weights (0.4/0.4/0.2 split); purely data-driven.
  - TF-IDF disadvantage: cold-start — rare class terms score low until enough examples exist.
''')

TF-IDF matrix: 29 docs × 300 features

Query text (truncated): checkout svc edge lb payment svc downstream payment error rate latency p99 ms latency p99 ms request drop rate p99 latency ms upstream 5xx rate upstream 5xx rate db connection pool used ratio db conne...

Top-3 similar incidents (TF-IDF cosine):
  [0.3393] INC-2025-09-05 — connection_pool_exhaustion on payment-svc
           Payment timeout 100%. Deploy v2.6 leak DB connection — pool from 50 → 50 hold, never retur

  [0.2793] INC-2025-11-08 — connection_pool_exhaustion on payment-svc
           Payment-svc v3.2 deploy at 09:42 leak DB pool. Pool 50/50 used trong 5 phút. Downstream ch

  [0.2774] INC-2026-03-20 — ddos on edge-lb
           Volumetric DDoS 5x normal traffic. Edge-lb saturate, all upstream visible degraded.

Comparison — keyword heuristic vs TF-IDF:
  Keyword top-1 : INC-2025-11-08 — connection_pool_exhaustion
  TF-IDF  top-1 : INC-2025-09-05 — connection_pool_exhaustion
  Keyword top-3 IDs : ['INC-2025-11-08'

---
# BONUS 3 — LLM Enrichment via Groq (Free Tier)

Thay Step 6 bằng LLM call: cung cấp graph RCA top-3 candidates, top similar incidents,
cluster alert fingerprints → LLM tổng hợp `root_cause_class`, `reasoning`, và `actions`.

**Reference**: [Anthropic "Building Effective Agents"](https://www.anthropic.com/research/building-effective-agents)
Ở đây chúng ta dùng pattern **Augmented LLM** (single LLM call + retrieval context)
thay vì full agentic loop — phù hợp cho RCA vì latency phải thấp trong incident response.

**Model**: `llama-3.3-70b-versatile` trên Groq free tier (nhanh, miễn phí).



In [12]:
import os
import urllib.request

# ── Load API key an toàn ──────────────────────────────────────────────────────
# Cách 1: set biến môi trường trước khi chạy notebook:
#   Windows PowerShell : $env:GROQ_API_KEY = 'gsk_...'
#   Windows CMD        : set GROQ_API_KEY=gsk_...
#   macOS/Linux        : export GROQ_API_KEY=gsk_...
# Cách 2: dùng file .env (thêm .env vào .gitignore, KHÔNG commit key)
#   pip install python-dotenv
#   from dotenv import load_dotenv; load_dotenv()
GROQ_API_KEY = 'Bỏ Key Vào Đây Lmao'
if not GROQ_API_KEY:
    raise EnvironmentError(
        'GROQ_API_KEY not set. '
        'Run: $env:GROQ_API_KEY = "gsk_..." in PowerShell trước khi chạy notebook.'
    )
print(f'Groq key loaded: {GROQ_API_KEY[:8]}...{GROQ_API_KEY[-4:]}  ')


# ── Prompt builder ────────────────────────────────────────────────────────────
def build_rca_prompt(cluster, graph_top3, similar_incidents):
    """Build structured prompt for LLM RCA enrichment."""
    top3_str = '\n'.join(
        f'  {rank+1}. {svc} (score={score:.4f})'
        for rank, (svc, score) in enumerate(graph_top3)
    )
    similar_str = ''
    for inc, score in similar_incidents:
        similar_str += (
            f'  - {inc["id"]} [{inc["severity"]}] {inc["root_cause_class"]} '
            f'on {inc["root_cause_service"]}\n'
            f'    Summary: {inc["summary"]}\n'
            f'    Remediation: {inc["remediation"]}\n'
        )
    fingerprints_str = '\n'.join(f'  - {fp}' for fp in cluster['fingerprints'][:10])

    return f"""You are an expert SRE performing automated root cause analysis.

## Cluster: {cluster['cluster_id']}
Time window: {cluster['first_ts']} to {cluster['last_ts']}
Max severity: {cluster['severity_max']}
Services in cluster: {', '.join(cluster['services'])}

## Graph + Temporal Scoring (top candidates)
{top3_str}

## Alert Fingerprints (sample)
{fingerprints_str}

## Most Similar Historical Incidents
{similar_str}

## Task
Based on the graph scoring, alert fingerprints, and historical incidents above, produce a JSON object with exactly these fields:
{{
  "root_cause": "<service name - must be one of the services listed above>",
  "class": "<one of: connection_pool_exhaustion, slow_query, memory_leak, rebalance_storm, deadlock, network_partition, bad_deploy, config_push, tls_expiry, ddos, thread_starvation, other>",
  "confidence": <float 0.0-1.0>,
  "reasoning": "<2-3 sentences explaining WHY this is the root cause, referencing specific signals>",
  "actions": ["<action 1>", "<action 2>", "<action 3>"]
}}

Return ONLY the JSON object, no markdown fences, no explanation outside the JSON."""


# ── Groq API caller ───────────────────────────────────────────────────────────
def call_groq_rca(prompt, api_key, model='llama-3.3-70b-versatile'):
    """Call Groq API (OpenAI-compatible endpoint) for LLM-enriched RCA."""
    payload = json.dumps({
        'model': model,
        'messages': [{'role': 'user', 'content': prompt}],
        'max_tokens': 1000,
        'temperature': 0,        # deterministic output
        'response_format': {'type': 'json_object'},  # force JSON mode
    }).encode('utf-8')

    req = urllib.request.Request(
    'https://api.groq.com/openai/v1/chat/completions',
    data=payload,
    headers={
        'Content-Type' : 'application/json',
        'Authorization': f'Bearer {api_key}',
        'User-Agent'   : 'python-requests/2.31.0',  # ← thêm dòng này
    },
    method='POST'
)
    with urllib.request.urlopen(req, timeout=30) as resp:
        data = json.loads(resp.read())

    raw_text = data['choices'][0]['message']['content']
    # Strip accidental markdown fences just in case
    clean = raw_text.strip().lstrip('```json').lstrip('```').rstrip('```').strip()
    return json.loads(clean)


# ── Run LLM enrichment for each cluster ──────────────────────────────────────
print('Running LLM-enriched RCA (Groq)...\n')

llm_results = []
for cluster in clusters:
    ranked    = rca_combined(cluster, G)
    graph_top3 = ranked[:3]
    sim_incs   = retrieve_similar_incidents(cluster, history, top_k=3)
    prompt     = build_rca_prompt(cluster, graph_top3, sim_incs)

    try:
        llm_out = call_groq_rca(prompt, GROQ_API_KEY)

        result = {
            'cluster_id'       : cluster['cluster_id'],
            'graph_top3'       : [[svc, score] for svc, score in graph_top3],
            'root_cause'       : llm_out.get('root_cause', ranked[0][0]),
            'class'            : llm_out.get('class', 'other'),
            'confidence'       : round(float(llm_out.get('confidence', ranked[0][1])), 4),
            'actions'          : llm_out.get('actions', ['Investigate manually']),
            'reasoning'        : llm_out.get('reasoning', ''),
            'similar_incidents': [inc['id'] for inc, _ in sim_incs],
            'method'           : 'graph+llm',
        }

        print(f"=== {cluster['cluster_id']} (Groq LLM) ===")
        print(f"  Root cause : {result['root_cause']}")
        print(f"  Class      : {result['class']}")
        print(f"  Confidence : {result['confidence']}")
        print(f"  Actions    : {result['actions']}")
        print(f"  Reasoning  : {result['reasoning']}")
        print()

        llm_results.append(result)

    except Exception as e:
        print(f'  ⚠ LLM call failed: {e}')
        print('  → Falling back to keyword retrieval result')
        llm_results.append(all_results[clusters.index(cluster)])


# ── Compare LLM vs kNN top-1 ──────────────────────────────────────────────────
print('\n=== Bonus 3 Comparison: kNN top-1 vs Groq LLM ===')
for i, cluster in enumerate(clusters):
    knn = all_results[i]
    llm = llm_results[i]
    print(f'Cluster {cluster["cluster_id"]}')
    print(f'  kNN  → root_cause={knn["root_cause"]}, class={knn["class"]}')
    print(f'  LLM  → root_cause={llm["root_cause"]}, class={llm["class"]}')
    agree = ' agree' if knn['class'] == llm['class'] else ' differ'
    print(f'  Class agreement: {agree}')

Groq key loaded: gsk_HVb7...Uo7v  
Running LLM-enriched RCA (Groq)...

=== c-000-000 (Groq LLM) ===
  Root cause : payment-svc
  Class      : connection_pool_exhaustion
  Confidence : 0.85
  Actions    : ['Rollback to a previous version of payment-svc', 'Scale the database connection pool to increase capacity', 'Implement monitoring and alerting for database connection pool usage']
  Reasoning  : The high graph score for payment-svc, combined with the crit alert for db_connection_pool_used_ratio and historical incidents of connection pool exhaustion, suggest that this is the root cause. The downstream_payment_error_rate and latency_p99_ms alerts on checkout-svc also support this conclusion, as they are likely symptoms of the payment service's connection pool issue. The similarity to past incidents INC-2025-11-08 and INC-2025-09-05 further increases confidence in this diagnosis.

=== c-000-001 (Groq LLM) ===
  Root cause : cart-svc
  Class      : memory_leak
  Confidence : 0.8
  Actions

In [16]:
# Use LLM results as the primary output (falls back to kNN if LLM failed)
final_output = {
    'clusters_analyzed': len(clusters),
    'results': llm_results
}

with open('results/rca_output_bonus.json', 'w', encoding='utf-8') as f:
    json.dump(final_output, f, indent=2, ensure_ascii=False)

print(' Written results/rca_output_bonus.json (method=graph+llm)')
print()
print(json.dumps(final_output, indent=2, ensure_ascii=False))

 Written results/rca_output_bonus.json (method=graph+llm)

{
  "clusters_analyzed": 5,
  "results": [
    {
      "cluster_id": "c-000-000",
      "graph_top3": [
        [
          "payment-svc",
          0.8166
        ],
        [
          "checkout-svc",
          0.5279
        ],
        [
          "edge-lb",
          0.45
        ]
      ],
      "root_cause": "payment-svc",
      "class": "connection_pool_exhaustion",
      "confidence": 0.85,
      "actions": [
        "Rollback to a previous version of payment-svc",
        "Scale the database connection pool to increase capacity",
        "Implement monitoring and alerting for database connection pool usage"
      ],
      "reasoning": "The high graph score for payment-svc, combined with the crit alert for db_connection_pool_used_ratio and historical incidents of connection pool exhaustion, suggest that this is the root cause. The downstream_payment_error_rate and latency_p99_ms alerts on checkout-svc also support this 